# ETL Raw - Silver

**Dataset:** Uber Ride Analytics - Nova Delhi, Índia  
**Fonte:** https://www.kaggle.com/datasets/yashdevladdha/uber-ride-analytics-dashboard

### Importações e Configurações do Postgres


In [32]:
import pandas as pd
import numpy as np
import psycopg2
import os
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from datetime import time

DB_NAME = "uber_analytics"
SCHEMA_NAME = "silver"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"

### Extração dos dados:

In [33]:
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
csv_path = os.path.join(
    project_root,
    "Data Layer",
    "raw",
    "dados_brutos.csv"
)
df = pd.read_csv(csv_path)

print(f"Total de registros: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")
print(f"Colunas: {df.columns}")

Total de registros: 150000
Total de colunas: 21
Colunas: Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method'],
      dtype='str')


### Mapeamento de Localizações para Regiões

In [34]:
LOCATION_TO_ZONE = {
    "Connaught Place": "Central Delhi",
    "Chandni Chowk": "Central Delhi",
    "Karol Bagh": "Central Delhi",
    "Paharganj": "Central Delhi",
    "Rajiv Chowk": "Central Delhi",
    "Barakhamba Road": "Central Delhi",
    "Janpath": "Central Delhi",
    "Kashmere Gate": "Central Delhi",
    "Mandi House": "Central Delhi",
    "Pragati Maidan": "Central Delhi",
    "Lal Quila": "Central Delhi",
    "ITO": "Central Delhi",
    "Jama Masjid": "Central Delhi",
    "Delhi Gate": "Central Delhi",
    "New Delhi Railway Station": "Central Delhi",
    "Dwarka": "West Delhi",
    "Janakpuri": "West Delhi",
    "Punjabi Bagh": "West Delhi",
    "Rajouri Garden": "West Delhi",
    "Tilak Nagar": "West Delhi",
    "Vikaspuri": "West Delhi",
    "Uttam Nagar": "West Delhi",
    "Moti Nagar": "West Delhi",
    "Kirti Nagar": "West Delhi",
    "Peeragarhi": "West Delhi",
    "Paschim Vihar": "West Delhi",
    "Ramesh Nagar": "West Delhi",
    "Subhash Nagar": "West Delhi",
    "Tagore Garden": "West Delhi",
    "Dwarka Mor": "West Delhi",
    "Dwarka Sector 21": "West Delhi",
    "Najafgarh": "West Delhi",
    "Nangloi": "West Delhi",
    "Madipur": "West Delhi",
    "Nawada": "West Delhi",
    "Mundka": "West Delhi",
    "Rohini": "North Delhi",
    "Pitampura": "North Delhi",
    "Shalimar Bagh": "North Delhi",
    "Model Town": "North Delhi",
    "Azadpur": "North Delhi",
    "GTB Nagar": "North Delhi",
    "Kamla Nagar": "North Delhi",
    "Netaji Subhash Place": "North Delhi",
    "Rohini East": "North Delhi",
    "Rohini West": "North Delhi",
    "Rithala": "North Delhi",
    "Samaypur Badli": "North Delhi",
    "Ashok Vihar": "North Delhi",
    "Jahangirpuri": "North Delhi",
    "Adarsh Nagar": "North Delhi",
    "Pulbangash": "North Delhi",
    "Tis Hazari": "North Delhi",
    "Vidhan Sabha": "North Delhi",
    "Civil Lines": "North Delhi",
    "Vishwavidyalaya": "North Delhi",
    "Kanhaiya Nagar": "North Delhi",
    "Kashmere Gate ISBT": "North Delhi",
    "Ashok Park Main": "North Delhi",
    "Inderlok": "North Delhi",
    "Keshav Puram": "North Delhi",
    "Lajpat Nagar": "South Delhi",
    "Saket": "South Delhi",
    "Hauz Khas": "South Delhi",
    "Green Park": "South Delhi",
    "Greater Kailash": "South Delhi",
    "Nehru Place": "South Delhi",
    "Kalkaji": "South Delhi",
    "Okhla": "South Delhi",
    "Malviya Nagar": "South Delhi",
    "South Extension": "South Delhi",
    "Defence Colony": "South Delhi",
    "Sarojini Nagar": "South Delhi",
    "RK Puram": "South Delhi",
    "Vasant Kunj": "South Delhi",
    "Munirka": "South Delhi",
    "IIT Delhi": "South Delhi",
    "Hauz Rani": "South Delhi",
    "Chirag Delhi": "South Delhi",
    "Govindpuri": "South Delhi",
    "Tughlakabad": "South Delhi",
    "Badarpur": "South Delhi",
    "Saket A Block": "South Delhi",
    "Saidulajab": "South Delhi",
    "Mehrauli": "South Delhi",
    "Qutub Minar": "South Delhi",
    "Chattarpur": "South Delhi",
    "Sultanpur": "South Delhi",
    "Ghitorni": "South Delhi",
    "Arjangarh": "South Delhi",
    "Aya Nagar": "South Delhi",
    "Maidan Garhi": "South Delhi",
    "IGNOU Road": "South Delhi",
    "New Colony": "South Delhi",
    "AIIMS": "South Delhi",
    "Moolchand": "South Delhi",
    "Panchsheel Park": "South Delhi",
    "Jasola": "South Delhi",
    "Chhatarpur": "South Delhi",
    "Ghitorni Village": "South Delhi",
    "Laxmi Nagar": "East Delhi",
    "Preet Vihar": "East Delhi",
    "Mayur Vihar": "East Delhi",
    "Nirman Vihar": "East Delhi",
    "Anand Vihar": "East Delhi",
    "Karkarduma": "East Delhi",
    "Shahdara": "East Delhi",
    "Dilshad Garden": "East Delhi",
    "Jhilmil": "East Delhi",
    "Mandawali": "East Delhi",
    "IP Extension": "East Delhi",
    "Patparganj": "East Delhi",
    "Yamuna Bank": "East Delhi",
    "Akshardham": "East Delhi",
    "Noida Mor": "East Delhi",
    "Kaushambi": "East Delhi",
    "Vaishali": "East Delhi",
    "Welcome": "East Delhi",
    "Seelampur": "East Delhi",
    "Shastri Park": "East Delhi",
    "Sarai Kale Khan": "East Delhi",
    "Vinobapuri": "East Delhi",
    "Ashram": "East Delhi",
    "Nizamuddin": "East Delhi",
    "Jangpura": "East Delhi",
    "Mansarovar Park": "East Delhi",
    "Indraprastha": "East Delhi",
    "Rajpath": "New Delhi",
    "India Gate": "New Delhi",
    "Chanakyapuri": "New Delhi",
    "Diplomatic Enclave": "New Delhi",
    "Bhikaji Cama Place": "New Delhi",
    "INA Market": "New Delhi",
    "Dhaula Kuan": "New Delhi",
    "Satguru Ram Singh Marg": "New Delhi",
    "Udyog Bhawan": "New Delhi",
    "Khan Market": "New Delhi",
    "Lodhi Road": "New Delhi",
    "Jor Bagh": "New Delhi",
    "Shivaji Park": "New Delhi",
    "Lok Kalyan Marg": "New Delhi",
    "Patel Chowk": "New Delhi",
    "Central Secretariat": "New Delhi",
    "Gurgaon": "NCR-Gurgaon",
    "Cyber Hub": "NCR-Gurgaon",
    "DLF Phase 1": "NCR-Gurgaon",
    "DLF Phase 2": "NCR-Gurgaon",
    "DLF Phase 3": "NCR-Gurgaon",
    "DLF Phase 4": "NCR-Gurgaon",
    "DLF Phase 5": "NCR-Gurgaon",
    "Golf Course Road": "NCR-Gurgaon",
    "Sohna Road": "NCR-Gurgaon",
    "MG Road": "NCR-Gurgaon",
    "IFFCO Chowk": "NCR-Gurgaon",
    "Huda City Centre": "NCR-Gurgaon",
    "Sikanderpur": "NCR-Gurgaon",
    "Udyog Vihar": "NCR-Gurgaon",
    "Udyog Vihar Phase 4": "NCR-Gurgaon",
    "Hero Honda Chowk": "NCR-Gurgaon",
    "Subhash Chowk": "NCR-Gurgaon",
    "Sadar Bazar Gurgaon": "NCR-Gurgaon",
    "Sushant Lok": "NCR-Gurgaon",
    "Vatika Chowk": "NCR-Gurgaon",
    "Manesar": "NCR-Gurgaon",
    "Badshahpur": "NCR-Gurgaon",
    "Gurgaon Sector 29": "NCR-Gurgaon",
    "Bhiwadi": "NCR-Gurgaon",
    "Bahadurgarh": "NCR-Gurgaon",
    "Khandsa": "NCR-Gurgaon",
    "Basai Dhankot": "NCR-Gurgaon",
    "Gurgaon Railway Station": "NCR-Gurgaon",
    "Narsinghpur": "NCR-Gurgaon",
    "Pataudi Chowk": "NCR-Gurgaon",
    "Ardee City": "NCR-Gurgaon",
    "Gurgaon Sector 56": "NCR-Gurgaon",
    "Ambience Mall": "NCR-Gurgaon",
    "DLF City Court": "NCR-Gurgaon",
    "Gwal Pahari": "NCR-Gurgaon",
    "IMT Manesar": "NCR-Gurgaon",
    "Civil Lines Gurgaon": "NCR-Gurgaon",
    "Kherki Daula Toll": "NCR-Gurgaon",
    "Palam Vihar": "NCR-Gurgaon",
    "Old Gurgaon": "NCR-Gurgaon",
    "Noida": "NCR-Noida",
    "Noida Sector 15": "NCR-Noida",
    "Noida Sector 16": "NCR-Noida",
    "Noida Sector 18": "NCR-Noida",
    "Noida Sector 62": "NCR-Noida",
    "Greater Noida": "NCR-Noida",
    "Film City": "NCR-Noida",
    "Botanical Garden": "NCR-Noida",
    "Golf Course Noida": "NCR-Noida",
    "Raj Nagar Extension": "NCR-Noida",
    "Indirapuram": "NCR-Noida",
    "Raj Nagar": "NCR-Noida",
    "Rajiv Nagar": "NCR-Noida",
    "Kadarpur": "NCR-Noida",
    "Shastri Nagar": "NCR-Noida",
    "Meerut": "NCR-Noida",
    "Ghaziabad": "NCR-Noida",
    "Noida Film City": "NCR-Noida",
    "Noida Extension": "NCR-Noida",
    "Anand Vihar ISBT": "NCR-Noida",
    "Faridabad": "NCR-Faridabad",
    "Faridabad Sector 16": "NCR-Faridabad",
    "Faridabad Sector 21": "NCR-Faridabad",
    "Ballabhgarh": "NCR-Faridabad",
    "Sonipat": "NCR-Faridabad",
    "Faridabad Sector 15": "NCR-Faridabad",
    "Panipat": "NCR-Faridabad",
    "IGI Airport": "Airport Zone",
    "IGI Airport T1": "Airport Zone",
    "IGI Airport T2": "Airport Zone",
    "IGI Airport T3": "Airport Zone",
    "Aerocity": "Airport Zone",
    "Mahipalpur": "Airport Zone"
}

ZONE_TO_REGION = {
    "North Delhi": "Delhi",
    "South Delhi": "Delhi",
    "East Delhi": "Delhi",
    "West Delhi": "Delhi",
    "Central Delhi": "Delhi",
    "New Delhi": "Delhi",
    "NCR-Gurgaon": "NCR",
    "NCR-Noida": "NCR",
    "NCR-Faridabad": "NCR",
    "Airport Zone": "Special Zone"
}

def get_zone(location):
    return LOCATION_TO_ZONE.get(location, "Unknown")

def get_region(zone):
    return ZONE_TO_REGION.get(zone, "Unknown")

print('Mapeamento carregado!')

Mapeamento carregado!


## Transformar

### Renomeando e preparando colunas

In [35]:
df_tratado = df.copy()
df_tratado = df_tratado.rename(columns={
    'Booking ID': 'id',
    'Booking Status': 'status',
    'Vehicle Type': 'vehicle',
    'Pickup Location': 'pickup',
    'Drop Location': 'drop',
    'Avg VTAT': 'vtat',
    'Avg CTAT': 'ctat',
    'Customer ID': 'customer_id',
    'Cancelled Rides by Customer': 'cancelled_by_customer',
    'Cancelled Rides by Driver': 'cancelled_by_driver',
    'Reason for cancelling by Customer': 'reason_cancelled_by_customer',
    'Driver Cancellation Reason': 'reason_cancelled_by_driver',
    'Incomplete Rides': 'incomplete',
    'Incomplete Rides Reason': 'reason_incomplete',
    'Booking Value': 'value',
    'Ride Distance': 'distance',
    'Payment Method': 'payment',
    'Customer Rating': 'customer_rating',
    'Driver Ratings': 'driver_rating',
    'Date': 'date',
    'Time': 'time',
})

if 'date' in df_tratado.columns:
    df_tratado['date'] = pd.to_datetime(df_tratado['date'], errors='coerce')
if 'time' in df_tratado.columns:
    df_tratado['time'] = pd.to_datetime(df_tratado['time'], format='%H:%M:%S', errors='coerce').dt.time

numeric_columns = ['vtat', 'ctat', 'value', 'distance', 'driver_rating', 'customer_rating']
for col in numeric_columns:
    if col in df_tratado.columns:
        df_tratado[col] = pd.to_numeric(df_tratado[col], errors='coerce')

boolean_columns = ['cancelled_by_customer', 'cancelled_by_driver', 'incomplete']
for col in boolean_columns:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].map({
            'TRUE': True, 'True': True, 'true': True, 1: True, '1': True,
            'FALSE': False, 'False': False, 'false': False, 0: False, '0': False
        })

### Removendo duplicatas e tratando nulos

In [36]:

df_tratado = df_tratado.drop_duplicates(subset=['id'], keep='first')
df_tratado = df_tratado.dropna(subset=['id'])
df_tratado = df_tratado.dropna(subset=['customer_id'])

for col in boolean_columns:
    if col in df_tratado.columns:
        df_tratado[col] = (
            df_tratado[col]
            .astype("boolean")
            .fillna(False)
            .astype(bool)
        )

text_columns = [
    'reason_cancelled_by_customer',
    'reason_cancelled_by_driver',
    'reason_incomplete'
]
for col in text_columns:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].fillna('')


### Validação de dados

In [37]:
if 'driver_rating' in df_tratado.columns:
    df_tratado.loc[(df_tratado['driver_rating'] < 1) | (df_tratado['driver_rating'] > 5), 'driver_rating'] = np.nan

if 'customer_rating' in df_tratado.columns:
    df_tratado.loc[(df_tratado['customer_rating'] < 1) | (df_tratado['customer_rating'] > 5), 'customer_rating'] = np.nan

if 'value' in df_tratado.columns:
    df_tratado.loc[df_tratado['value'] < 0, 'value'] = np.nan

if 'distance' in df_tratado.columns:
    df_tratado.loc[df_tratado['distance'] < 0, 'distance'] = 0

### Padronização de categorias

In [38]:
if 'status' in df_tratado.columns:
    df_tratado['status'] = df_tratado['status'].str.strip().str.title()

if 'vehicle' in df_tratado.columns:
    df_tratado['vehicle'] = df_tratado['vehicle'].str.strip().str.title()

if 'payment' in df_tratado.columns:
    df_tratado['payment'] = df_tratado['payment'].str.strip().str.title()

for col in ['pickup', 'drop']:
    if col in df_tratado.columns:
        df_tratado[col] = df_tratado[col].str.strip().str.title()

### Enriquecimento Geográfico

In [39]:
df_tratado['pickup_zone'] = df_tratado['pickup'].apply(get_zone)
df_tratado['pickup_region'] = df_tratado['pickup_zone'].apply(get_region)
df_tratado['drop_zone'] = df_tratado['drop'].apply(get_zone)
df_tratado['drop_region'] = df_tratado['drop_zone'].apply(get_region)

print(f'Zonas únicas: {df_tratado["pickup_zone"].nunique()}')
print(f'Regiões únicas: {df_tratado["pickup_region"].nunique()}')

Zonas únicas: 10
Regiões únicas: 3


### Categorização de Distância

In [40]:
def categorize_distance(distance):
    if pd.isna(distance):
        return None
    if distance < 10:
        return 'Curta'
    elif distance <= 25:
        return 'Média'
    else:
        return 'Longa'

df_tratado['distance_category'] = df_tratado['distance'].apply(categorize_distance)

df_tratado['value_per_km'] = None
mask = (df_tratado['status'] == 'Completed') & (df_tratado['distance'] > 0) & (df_tratado['value'].notna())
df_tratado.loc[mask, 'value_per_km'] = df_tratado.loc[mask, 'value'] / df_tratado.loc[mask, 'distance']

print('Categorização concluída!')

Categorização concluída!


### Selecionando apenas as colunas desejadas( caso haja atualização da base de dados)

In [41]:
colunas_finais = [
    'id', 'date', 'time', 'status', 'customer_id', 'vehicle',
    'pickup', 'drop',
    'pickup_zone', 'pickup_region', 'drop_zone', 'drop_region',
    'vtat', 'ctat',
    'cancelled_by_customer', 'reason_cancelled_by_customer',
    'cancelled_by_driver', 'reason_cancelled_by_driver',
    'incomplete', 'reason_incomplete',
    'value', 'distance', 'distance_category', 'value_per_km',
    'driver_rating', 'customer_rating', 'payment'
]

colunas_existentes = [col for col in colunas_finais if col in df_tratado.columns]
df_final = df_tratado[colunas_existentes].copy()

print(f"Total de registros: {len(df_final)}")
print(f"Total de colunas: {len(df_final.columns)}")

Total de registros: 148767
Total de colunas: 27


## Carregar no PostgreSQL

### Criando e Conectando com o banco PostgreSQL

In [42]:
def criar_banco_e_schema_se_nao_existir():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT,
            database='postgres'
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()
        cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'")
        existe = cursor.fetchone()
        if not existe:
            cursor.execute(f"CREATE DATABASE {DB_NAME}")
            print(f"Banco de dados '{DB_NAME}' criado com sucesso!")
        else:
            print(f"Banco de dados '{DB_NAME}' ja existe.")
        cursor.close()
        conn.close()
        conn = psycopg2.connect(
            host=DB_HOST,
            user=DB_USER,
            password=DB_PASSWORD,
            port=DB_PORT,
            database=DB_NAME
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()
        cursor.execute(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
        print(f"Schema '{SCHEMA_NAME}' criado/verificado com sucesso!")
        cursor.close()
        conn.close()
    except Exception as e:
        print(f"Erro ao criar banco/schema: {e}")
        raise

criar_banco_e_schema_se_nao_existir()

Banco de dados 'uber_analytics' ja existe.
Schema 'silver' criado/verificado com sucesso!


### Criando tabela BOOKING no PostgreSQL

In [43]:
conexao = psycopg2.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    port=DB_PORT,
    database=DB_NAME
)
conexao.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cursor = conexao.cursor()

create_table_sql = f"""
CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME};

CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.booking (
    id VARCHAR(50) PRIMARY KEY,
    date DATE,
    time TIME,
    status VARCHAR(50),
    customer_id VARCHAR(50),
    vehicle VARCHAR(50),
    pickup VARCHAR(100),
    drop VARCHAR(100),
    pickup_zone VARCHAR(50),
    pickup_region VARCHAR(50),
    drop_zone VARCHAR(50),
    drop_region VARCHAR(50),
    vtat DECIMAL(10,2),
    ctat DECIMAL(10,2),
    cancelled_by_customer BOOLEAN,
    reason_cancelled_by_customer TEXT,
    cancelled_by_driver BOOLEAN,
    reason_cancelled_by_driver TEXT,
    incomplete BOOLEAN,
    reason_incomplete TEXT,
    value DECIMAL(10,2),
    distance DECIMAL(10,2),
    distance_category VARCHAR(20),
    value_per_km DECIMAL(10,2),
    driver_rating DECIMAL(3,2),
    customer_rating DECIMAL(3,2),
    payment VARCHAR(50)
);
"""

try:
    cursor.execute(create_table_sql)
    conexao.commit()
    print(f"Tabela '{SCHEMA_NAME}.booking' criada com sucesso!")
except Exception as e:
    print(f"Erro: {e}")
    conexao.rollback()

Tabela 'silver.booking' criada com sucesso!


### Limpando tabela caso exista

In [44]:
try:
    cursor.execute(f"DELETE FROM {SCHEMA_NAME}.booking;")
    conexao.commit()
except Exception as e:
    print(f"Tabela {e} já vazia")
    conexao.rollback()

### Preparando dados para inserção

In [45]:
def preparar_valor(valor):
    if pd.isna(valor):
        return None
    elif isinstance(valor, bool):
        return bool(valor)
    elif isinstance(valor, (int, np.integer)):
        return int(valor)
    elif isinstance(valor, (float, np.floating)):
        return float(valor)
    elif isinstance(valor, pd.Timestamp):
        return valor.strftime('%Y-%m-%d')
    elif isinstance(valor, time):
        return valor.strftime('%H:%M:%S')
    else:
        return str(valor)

dados_para_inserir = []

for _, row in df_final.iterrows():
    valores = []
    for col in df_final.columns:
        valores.append(preparar_valor(row[col]) if col in row else None)
    dados_para_inserir.append(tuple(valores))

print(f"\nDados preparados: {len(dados_para_inserir):,} registros prontos para inserção")


Dados preparados: 148,767 registros prontos para inserção


### Inserindo dados em lote

In [46]:
placeholders = ', '.join(['%s'] * len(df_final.columns))
colunas_sql = ', '.join(df_final.columns)
insert_sql = f"INSERT INTO {SCHEMA_NAME}.booking ({colunas_sql}) VALUES ({placeholders})"

batch_size = 1000
total_inseridos = 0
for i in range(0, len(dados_para_inserir), batch_size):
    batch = dados_para_inserir[i:i + batch_size]
    cursor.executemany(insert_sql, batch)
    conexao.commit()
    total_inseridos += len(batch)

    if (i + batch_size) % 5000 == 0 or i + batch_size >= len(dados_para_inserir):
        print(f"  Inseridos: {total_inseridos:,} / {len(dados_para_inserir):,} registros...")

  Inseridos: 5,000 / 148,767 registros...
  Inseridos: 10,000 / 148,767 registros...
  Inseridos: 15,000 / 148,767 registros...
  Inseridos: 20,000 / 148,767 registros...
  Inseridos: 25,000 / 148,767 registros...
  Inseridos: 30,000 / 148,767 registros...
  Inseridos: 35,000 / 148,767 registros...
  Inseridos: 40,000 / 148,767 registros...
  Inseridos: 45,000 / 148,767 registros...
  Inseridos: 50,000 / 148,767 registros...
  Inseridos: 55,000 / 148,767 registros...
  Inseridos: 60,000 / 148,767 registros...
  Inseridos: 65,000 / 148,767 registros...
  Inseridos: 70,000 / 148,767 registros...
  Inseridos: 75,000 / 148,767 registros...
  Inseridos: 80,000 / 148,767 registros...
  Inseridos: 85,000 / 148,767 registros...
  Inseridos: 90,000 / 148,767 registros...
  Inseridos: 95,000 / 148,767 registros...
  Inseridos: 100,000 / 148,767 registros...
  Inseridos: 105,000 / 148,767 registros...
  Inseridos: 110,000 / 148,767 registros...
  Inseridos: 115,000 / 148,767 registros...
  Inseri

### Fechando a conexão.

In [47]:

cursor.close()
conexao.close()

print(f"\nConexão fechada\n")



Conexão fechada

